# Amazon ML Challenge 2026 - Normalization Analysis

This notebook demonstrates the Phase 2 normalization pipeline and shows before/after examples.

**Core Normalization Principles:**
1. **Preservation of Raw Values:** The original raw fields are never overwritten. Normalized variants and token collections are stored side-by-side.
2. **Conservative & Train-Derived:** Normalization rules are conservative and derived strictly from observed training patterns.
3. **Open-Set Support:** Structured field normalizers (e.g. country) allow unseen values to pass through without being dropped.

In [1]:
from _pytest import pathlib
import sys
import json
from pathlib import Path
import pandas as pd

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# Load diagnostics
diag_path = Path("output/normalization_diagnostics.json")
if not diag_path.exists():
    diag_path = Path("../output/normalization_diagnostics.json")

with open(diag_path, "r", encoding="utf-8") as f:
    diag = json.load(f)
print("Loaded normalization diagnostics successfully.")

ModuleNotFoundError: No module named '_pytest'

In [ ]:
# Overall Statistics
print("=== Normalization Diagnostics Summary ===")
diag_rows = []
for src, fields in diag.items():
    for field, stats in fields.items():
        if field.endswith('_examples'):
            continue
        diag_rows.append({
            "Source": src,
            "Field": field,
            "Total Sampled": stats['total'],
            "Changed (%)": f"{stats['changed_pct']}%",
            "Empty Normalized (%)": f"{stats['empty_normalized_pct']}%",
            "Avg Tokens": stats['avg_tokens'],
            "Collisions": stats['collision_count']
        })
pd.DataFrame(diag_rows)

In [ ]:
# Business Name Examples
print("=== Business Name Normalization Examples ===")
for src, fields in diag.items():
    print(f"\n{src} Business Name Examples:")
    for ex in fields.get('business_name_examples', [])[:5]:
        print(f"  '{ex['raw']}'")
        print(f"  -> '{ex['normalized']}'")

In [ ]:
# Business Address Examples
print("=== Business Address Normalization Examples ===")
for src, fields in diag.items():
    print(f"\n{src} Business Address Examples:")
    for ex in fields.get('business_address_examples', [])[:5]:
        print(f"  '{ex['raw']}'")
        print(f"  -> '{ex['normalized']}'")

In [ ]:
# Test Normalization Pipeline
import sys
from pathlib import Path

for p in [Path("src"), Path("../src")]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
        break

from amazon_ml.normalization import EntityNormalizer

normalizer = EntityNormalizer()

# Test cases from real dataset
test_cases = [
    {
        'entity_id': 'S1-123',
        'business_name': "Acme & Co. LLC",
        'business_address': "123 Main St, Apt 4B",
        'country': "us"
    },
    {
        'entity_id': 'S2-456',
        'business_name': "Global Private Limited",
        'business_address': "KH NO. -570/13, NEW DELHI, WEST DELHI",
        'country': "india"
    },
    {
        'entity_id': 'S3-789',
        'business_name': "Test Corp.",
        'business_address': "456 Oak Ave, Suite 200",
        'country': "france"
    },
]

for tc in test_cases:
    result = normalizer.normalize_record(tc)
    print(f"\nEntity: {tc['entity_id']}")
    print(f"  Name: '{tc['business_name']}'")
    print(f"    -> raw preserved: '{result['business_name']['raw']}'")
    print(f"    -> field_specific: '{result['business_name']['field_specific_normalized']}'")
    print(f"    -> tokens: {result['business_name']['tokens']}")
    print(f"  Addr: '{tc['business_address']}'")
    print(f"    -> field_specific: '{result['business_address']['field_specific_normalized']}'")
    print(f"    -> tokens: {result['business_address']['tokens']}")
    print(f"  Country: '{tc['country']}'")
    print(f"    -> field_specific: '{result['country']['field_specific_normalized']}'")

In [ ]:
# Test Individual Normalizers
from amazon_ml.normalization import (
    BusinessNameNormalizer,
    AddressNormalizer,
    StructuredFieldNormalizer,
    Tokenizer,
)

bnn = BusinessNameNormalizer()
an = AddressNormalizer()
sfn = StructuredFieldNormalizer()
tok = Tokenizer()

# Business name normalization steps
name = "Acme & Co. LLC"
print(f"Original: {name}")
print(f"Ampersand: {bnn.normalize_ampersand(name)}")
print(f"Legal suffixes: {bnn.normalize_legal_suffixes(name)}")
print(f"Full: {bnn.full_normalize(name)}")
print(f"Tokens: {tok.normalized_tokens(bnn.full_normalize(name))}")
print(f"Char 3-grams: {tok.char_ngrams(bnn.full_normalize(name))[:10]}...")

In [ ]:
# Address normalization steps
addr = "123 Main St, Apt 4B, New York, NY"
print(f"Original: {addr}")
print(f"Punctuation: {an.normalize_punctuation(addr)}")
print(f"Street types: {an.normalize_street_types(addr)}")
print(f"Unit types: {an.normalize_unit_types(addr)}")
print(f"Directionals: {an.normalize_directionals(addr)}")
print(f"Full: {an.full_normalize(addr)}")
print(f"Tokens: {tok.normalized_tokens(an.full_normalize(addr))}")

In [ ]:
# Structured field normalization (Open-Set Country Support)
print("Country examples:")
for c in ['us', 'USA', 'india', 'france', 'germany', 'UK']:
    print(f"  {c} -> {sfn.normalize_country(c)}")

print("\nPostal code examples:")
for p in ['12345', '12345-6789', 'SW1A 1AA', '110001']:
    print(f"  {p} -> {sfn.normalize_postal_code(p)}")

print("\nPhone examples:")
for p in ['(123) 456-7890', '+1-123-456-7890', '123.456.7890']:
    print(f"  {p} -> {sfn.normalize_phone(p)}")

print("\nEmail examples:")
for e in ['TEST@EXAMPLE.COM', 'user@domain.com']:
    print(f"  {e} -> {sfn.normalize_email(e)}")

In [ ]:
# Collision Analysis
print("=== Normalization Collisions ===")
for src, fields in diag.items():
    for field in ['business_name', 'business_address', 'country']:
        if 'example_collisions' in fields[field]:
            collisions = fields[field]['example_collisions']
            if collisions:
                print(f"\n{src} {field} collisions ({len(collisions)} total):")
                for norm_val, raw_vals in list(collisions.items())[:3]:
                    print(f"  Normalized: '{norm_val}'")
                    for rv in raw_vals[:3]:
                        print(f"    From: '{rv}'")